In [1]:
"""
Sprint 5 — Modèles spécialisés par catégorie

Entraîne un NanoGPT dédié pour chaque (langue × type × secteur).
Résultat : 8 modèles FR + 8 modèles AR = 16 fichiers .pt
Nommage  : model_{lang}_{type}_{secteur}.pt
           ex: model_fr_marque_luxe.pt

À utiliser si le modèle unifié (Sprint3_v2) ne donne pas
assez de cohérence par secteur.
"""
import sys, os, json, torch, torch.optim as optim
import torch.nn.functional as F

BACKEND_PATH = os.path.join(os.getcwd(), '..', 'backend')
if BACKEND_PATH not in sys.path:
    sys.path.insert(0, BACKEND_PATH)

from app.models.nanogpt import NanoGPT

WEIGHTS_DIR = os.path.join(BACKEND_PATH, 'app', 'weights')
os.makedirs(WEIGHTS_DIR, exist_ok=True)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Périphérique : {device.upper()}')

Périphérique : CPU


In [6]:
def train_category_model(dataset_path, lang, type_, secteur,
                         max_iters=3000, batch_size=32,
                         block_size=24, lr=1e-3):
    """
    Entraîne et sauvegarde un modèle dédié pour une catégorie.

    Paramètres
    ----------
    dataset_path : chemin vers le .txt de la catégorie
    lang         : 'fr' ou 'ar'
    type_        : 'marque' ou 'societe'
    secteur      : 'luxe', 'tech', 'food', etc.
    """
    print(f"\n{'=' * 55}")
    print(f'  {lang.upper()} | {type_} | {secteur}')
    print(f"\n{'=' * 55}")

    # ── 1. Chargement ──────────────────────────────────────
    if not os.path.exists(dataset_path):
        print(f'  ⚠️  Fichier manquant : {dataset_path} — skip')
        return None

    with open(dataset_path, encoding='utf-8') as f:
        names = [l.strip() for l in f if l.strip() and ' ' not in l.strip()]

    if len(names) < 30:
        print(f'  ⚠️  Trop peu de données ({len(names)} noms) — skip')
        return None

    print(f'  Noms chargés : {len(names)}')

    # ── 2. Vocabulaire LOCAL à cette catégorie ─────────────
    # On lit aussi le vocab partagé pour inclure '.' comme séparateur
    text  = '\n'.join(names) + '\n'
    chars = sorted(list(set(text)))
    stoi  = {ch: i for i, ch in enumerate(chars)}
    itos  = {i: ch for i, ch in enumerate(chars)}
    vs    = len(chars)

    # ── 3. Tenseurs train/val ──────────────────────────────
    data       = torch.tensor([stoi[c] for c in text], dtype=torch.long)
    n          = int(0.9 * len(data))
    train_data = data[:n]
    val_data   = data[n:]

    # ── 4. Modèle ──────────────────────────────────────────
    model = NanoGPT(vocab_size=vs, n_embed=64, n_head=4,
                    n_layer=4, block_size=block_size, dropout=0.1)
    model.to(device)

    # ── 5. Entraînement ───────────────────────────────────
    optimizer = optim.AdamW(model.parameters(), lr=lr)

    def get_batch(split):
        d  = train_data if split == 'train' else val_data
        ix = torch.randint(max(1, len(d) - block_size), (batch_size,))
        x  = torch.stack([d[i:i+block_size]     for i in ix])
        y  = torch.stack([d[i+1:i+block_size+1] for i in ix])
        return x.to(device), y.to(device)

    model.train()
    for it in range(max_iters):
        if it % (max_iters // 4) == 0 or it == max_iters - 1:
            model.eval()
            with torch.no_grad():
                _, lt = model(*get_batch('train'))
                _, lv = model(*get_batch('val'))
            print(f'  Iter {it:5d} | Train {lt.item():.3f} | Val {lv.item():.3f}')
            model.train()
        xb, yb = get_batch('train')
        _, loss = model(xb, yb)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()

    # ── 6. Sauvegarde poids + vocab local ─────────────────
    tag       = f'{lang}_{type_}_{secteur}'
    pt_path   = os.path.join(WEIGHTS_DIR, f'model_{tag}.pt')
    json_path = os.path.join(WEIGHTS_DIR, f'vocab_{tag}.json')

    torch.save(model.to('cpu').state_dict(), pt_path)
    with open(json_path, 'w', encoding='utf-8') as jf:
        json.dump({'stoi': stoi}, jf, ensure_ascii=False, indent=2)

    print(f'  ✅ Sauvegardé : model_{tag}.pt + vocab_{tag}.json')
    return {'stoi': stoi, 'itos': itos, 'model': model}

print('Fonction train_category_model définie ✓')

Fonction train_category_model définie ✓


In [7]:
# ── Entraînement de tous les modèles FRANÇAIS ────────────────
FR_DATA = os.path.join(os.getcwd(), '..', 'data', 'fr')

CATEGORIES_FR = [
    ('marque',  'luxe'),
    ('marque',  'tech'),
    ('marque',  'food'),
    ('marque',  'general'),
    ('societe', 'tech'),
    ('societe', 'services'),
    ('societe', 'industrie'),
    ('societe', 'general'),
]

results_fr = {}
for type_, secteur in CATEGORIES_FR:
    path   = os.path.join(FR_DATA, f'{type_}_{secteur}.txt')
    result = train_category_model(
        dataset_path = path,
        lang         = 'fr',
        type_        = type_,
        secteur      = secteur,
        max_iters    = 3000,
        batch_size   = 32,
        block_size   = 24,
    )
    if result:
        results_fr[f'{type_}_{secteur}'] = result

print(f'\n✓ {len(results_fr)} modèles FR entraînés')


  FR | marque | luxe

  Noms chargés : 155
  Iter     0 | Train 3.234 | Val 3.246
  Iter   750 | Train 0.355 | Val 3.404
  Iter  1500 | Train 0.265 | Val 4.279
  Iter  2250 | Train 0.234 | Val 4.720
  Iter  2999 | Train 0.235 | Val 5.109
  ✅ Sauvegardé : model_fr_marque_luxe.pt + vocab_fr_marque_luxe.json

  FR | marque | tech

  Noms chargés : 160
  Iter     0 | Train 3.219 | Val 3.230
  Iter   750 | Train 0.355 | Val 3.492
  Iter  1500 | Train 0.260 | Val 4.005
  Iter  2250 | Train 0.215 | Val 4.379
  Iter  2999 | Train 0.225 | Val 4.263
  ✅ Sauvegardé : model_fr_marque_tech.pt + vocab_fr_marque_tech.json

  FR | marque | food

  Noms chargés : 118
  Iter     0 | Train 3.155 | Val 3.160
  Iter   750 | Train 0.316 | Val 3.432
  Iter  1500 | Train 0.224 | Val 4.383
  Iter  2250 | Train 0.201 | Val 4.839
  Iter  2999 | Train 0.181 | Val 4.628
  ✅ Sauvegardé : model_fr_marque_food.pt + vocab_fr_marque_food.json

  FR | marque | general

  Noms chargés : 196
  Iter     0 | Train 3.216 | 

In [8]:
AR_DATA = os.path.join(os.getcwd(), '..', 'data', 'ar')

CATEGORIES_AR = [
    ('marque',  'luxe'),
    ('marque',  'tech'),
    ('marque',  'food'),
    ('marque',  'general'),
    ('societe', 'tech'),
    ('societe', 'services'),
    ('societe', 'industrie'),
    ('societe', 'general'),
]

results_ar = {}
for type_, secteur in CATEGORIES_AR:
    path   = os.path.join(AR_DATA, f'{type_}_{secteur}.txt')
    result = train_category_model(
        dataset_path = path,
        lang         = 'ar',
        type_        = type_,
        secteur      = secteur,
        max_iters    = 2000,
        batch_size   = 32,
        block_size   = 16,   # plus court pour l'arabe
    )
    if result:
        results_ar[f'{type_}_{secteur}'] = result

print(f'\n✓ {len(results_ar)} modèles AR entraînés')


  AR | marque | luxe

  Noms chargés : 146
  Iter     0 | Train 3.555 | Val 3.550
  Iter   500 | Train 0.476 | Val 4.617
  Iter  1000 | Train 0.269 | Val 5.203
  Iter  1500 | Train 0.290 | Val 5.795
  Iter  1999 | Train 0.252 | Val 5.735
  ✅ Sauvegardé : model_ar_marque_luxe.pt + vocab_ar_marque_luxe.json

  AR | marque | tech

  Noms chargés : 107
  Iter     0 | Train 3.465 | Val 3.484
  Iter   500 | Train 0.384 | Val 4.565
  Iter  1000 | Train 0.277 | Val 5.667
  Iter  1500 | Train 0.265 | Val 5.920
  Iter  1999 | Train 0.234 | Val 6.403
  ✅ Sauvegardé : model_ar_marque_tech.pt + vocab_ar_marque_tech.json

  AR | marque | food

  Noms chargés : 105
  Iter     0 | Train 3.501 | Val 3.541
  Iter   500 | Train 0.334 | Val 5.329
  Iter  1000 | Train 0.256 | Val 6.005
  Iter  1500 | Train 0.220 | Val 6.720
  Iter  1999 | Train 0.231 | Val 7.293
  ✅ Sauvegardé : model_ar_marque_food.pt + vocab_ar_marque_food.json

  AR | marque | general

  Noms chargés : 105
  Iter     0 | Train 3.507 | 

In [9]:
def quick_generate(result, n=8, temperature=0.8, top_k=15):
    """Génération rapide depuis un result de train_category_model."""
    model = result['model'].to(device)
    stoi  = result['stoi']
    itos  = result['itos']
    model.eval()
    names = []
    pad   = stoi.get('\n', 0)
    for _ in range(n * 4):
        ctx = torch.zeros(1, model.block_size, dtype=torch.long).to(device)
        ctx[0, -1] = pad
        chars = []
        with torch.no_grad():
            for _ in range(18):
                logits, _ = model(ctx)
                logits = logits[:, -1, :] / max(temperature, 1e-5)
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = float('-inf')
                ix  = torch.multinomial(F.softmax(logits, -1), 1).item()
                ch  = itos.get(ix, '')
                if ch == '\n' or not ch:
                    break
                chars.append(ch)
                new = torch.roll(ctx, -1, dims=1).clone()
                new[0,-1] = ix
                ctx = new
        nm = ''.join(chars).strip()
        if 3 <= len(nm) <= 15 and nm not in names:
            names.append(nm)
        if len(names) >= n:
            break
    return names[:n]

# ── Test FR ─────────────────────────────────────────────────
print('=== Génération FR par modèle spécialisé ===')
for key, result in list(results_fr.items())[:4]:
    noms = quick_generate(result, n=6)
    print(f'  {key:25s} → {noms}')

# ── Test AR ─────────────────────────────────────────────────
print('\n=== Génération AR par modèle spécialisé ===')
for key, result in list(results_ar.items())[:4]:
    noms = quick_generate(result, n=5, temperature=0.9)
    print(f'  {key:25s} → {noms}')

=== Génération FR par modèle spécialisé ===
  marque_luxe               → ['soinorex', 'saphirex', 'divinalis', 'florival', 'royorex', 'puralis']
  marque_tech               → ['ioretech', 'iocraft', 'iocore', 'tecraft', 'cyptalis', 'approaxis']
  marque_food               → ['fruitalis', 'gourmalia', 'greenfood', 'bionappet', 'epicur', 'granalis']
  marque_general            → ['aroricor', 'oprimalis', 'aroxide', 'oprimox', 'ecoiprime', 'designex']

=== Génération AR par modèle spécialisé ===
  marque_luxe               → ['زاهية', 'إناء', 'لامية', 'سناء', 'زبرجد']
  marque_tech               → ['ابداعية', 'زخم', 'تقدم', 'القنطلاقة', 'اقتدار']
  marque_food               → ['زاكية', 'طاهرة', 'رزق', 'طبيعية', 'بستانية']
  marque_general            → ['اكرام', 'جودة', 'المجد', 'تعاون', 'اساس']


In [10]:
import os
print('=== Inventaire des fichiers poids générés ===')
files = sorted(os.listdir(WEIGHTS_DIR))
total_ko = 0
for f in files:
    fp = os.path.join(WEIGHTS_DIR, f)
    ko = os.path.getsize(fp) // 1024
    total_ko += ko
    tag = '🔶' if f.endswith('.pt') else '📄'
    print(f'  {tag}  {f:45s}  {ko:6d} Ko')
print(f'\n  TOTAL : {total_ko} Ko ({total_ko//1024} Mo)')
print(f'\nSprint 5 validé ✓ — {len([f for f in files if f.endswith(".pt")])} modèles disponibles')

=== Inventaire des fichiers poids générés ===
  🔶  model_ar.pt                                       847 Ko
  🔶  model_ar_marque_food.pt                           819 Ko
  🔶  model_ar_marque_general.pt                        820 Ko
  🔶  model_ar_marque_luxe.pt                           820 Ko
  🔶  model_ar_marque_tech.pt                           818 Ko
  🔶  model_ar_societe_general.pt                       819 Ko
  🔶  model_ar_societe_industrie.pt                     820 Ko
  🔶  model_ar_societe_services.pt                      820 Ko
  🔶  model_ar_societe_tech.pt                          818 Ko
  🔶  model_fr.pt                                       842 Ko
  🔶  model_fr_marque_food.pt                           822 Ko
  🔶  model_fr_marque_general.pt                        822 Ko
  🔶  model_fr_marque_luxe.pt                           822 Ko
  🔶  model_fr_marque_tech.pt                           822 Ko
  🔶  model_fr_societe_general.pt                       822 Ko
  🔶  model_fr_societe_in